# BioT5 Mini Post-Training Collection On Kaggle

This notebook clones the repo, ensures the ChEBI-20 dataset exists locally, runs the active BioT5 grouped collection pipeline on 128 train descriptions with 100 samples per description, derives grouped train/validation/test splits, and exports a small `mini-post-training.zip` artifact ready for post-training debugging.


In [ ]:
from pathlib import Path
import shutil
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet"
REPO_DIR = Path("/kaggle/working/Thesis")
STAGE_NAME = "mini_post_training"

%cd /kaggle/working
!if [ -d "{REPO_DIR / '.git'}" ]; then echo "Reusing {REPO_DIR}"; elif [ -d "{REPO_DIR}" ]; then echo "Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook." && false; else git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"; fi
!git -C "{REPO_DIR}" fetch --depth 1 origin "{REPO_BRANCH}" && git -C "{REPO_DIR}" checkout -B "{REPO_BRANCH}" FETCH_HEAD

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from thesis_kaggle_support import (
    ARTIFACT_EXPORT_ROOT,
    create_zip_archive,
    dump_yaml,
    ensure_grouped_split_files,
    ensure_paths_exist,
    ensure_runtime_dependencies,
    export_stage_artifacts,
    json_dumps,
    load_yaml,
    read_json,
    read_jsonl,
    report_runtime,
)

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

huggingface_login_status = "not_attempted"
access_token = None
try:
    access_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    huggingface_login_status = f"secret_unavailable:{exc.__class__.__name__}"

if access_token:
    login(token=access_token)
    huggingface_login_status = "logged_in"
elif huggingface_login_status == "not_attempted":
    huggingface_login_status = "no_huggingface_token"


In [ ]:
MAX_DESCRIPTIONS = 128
TARGET_MOLECULES_PER_DESCRIPTION = 100
NUM_PARTS = 1
PART_INDEX = 1
VALIDATION_FRACTION = 0.05
TEST_FRACTION = 0.05
SEED = 42

CHEBI_OUTPUT_DIR = REPO_DIR / "data" / "chebi20"
CHEBI_PROCESSED_DIR = CHEBI_OUTPUT_DIR / "processed"
COLLECTION_OUTPUT_DIR = REPO_DIR / "data_collection" / "outputs" / "kaggle_chebi20_biot5_train_mini"
DERIVED_TRAIN_FILE = REPO_DIR / "data" / "post_training" / "processed" / "train_multimol_mini_debug.jsonl"
SPLIT_OUTPUT_DIR = REPO_DIR / "kaggle" / "generated_data" / "mini_post_training_splits"
OUTPUT_DIR = REPO_DIR / "outputs" / "kaggle" / "mini_post_training"
CONFIG_SNAPSHOT_PATH = OUTPUT_DIR / "config_snapshot.json"
TEMP_CONFIG_PATH = REPO_DIR / "kaggle" / "generated_configs" / "collect_biot5_chebi20.mini_debug.kaggle.yaml"
ARTIFACT_ZIP_PATH = ARTIFACT_EXPORT_ROOT / "mini-post-training.zip"

if PART_INDEX < 1 or PART_INDEX > NUM_PARTS:
    raise ValueError(f"PART_INDEX must be between 1 and {NUM_PARTS}, got {PART_INDEX}")

ensure_runtime_dependencies(REPO_DIR)
runtime_report = report_runtime(require_gpu=True)
print(json_dumps({
    "runtime": runtime_report,
    "huggingface_login_status": huggingface_login_status,
    "stage_name": STAGE_NAME,
    "max_descriptions": MAX_DESCRIPTIONS,
    "target_molecules_per_description": TARGET_MOLECULES_PER_DESCRIPTION,
    "requested_raw_candidates": MAX_DESCRIPTIONS * TARGET_MOLECULES_PER_DESCRIPTION,
    "num_parts": NUM_PARTS,
    "part_index": PART_INDEX,
    "collection_output_dir": str(COLLECTION_OUTPUT_DIR),
    "derived_train_file": str(DERIVED_TRAIN_FILE),
    "split_output_dir": str(SPLIT_OUTPUT_DIR),
    "artifact_zip_path": str(ARTIFACT_ZIP_PATH),
}))


In [ ]:
have_processed = all((CHEBI_PROCESSED_DIR / f"{split}.jsonl").exists() for split in ("train", "validation", "test"))
print(json_dumps({
    "have_processed": have_processed,
    "processed_dir": str(CHEBI_PROCESSED_DIR),
}))

%cd {REPO_DIR}
!if [ -f "{CHEBI_PROCESSED_DIR / 'train.jsonl'}" ] && [ -f "{CHEBI_PROCESSED_DIR / 'validation.jsonl'}" ] && [ -f "{CHEBI_PROCESSED_DIR / 'test.jsonl'}" ]; then echo "ChEBI processed splits already exist"; else python scripts/download_chebi20.py --output-dir "{CHEBI_OUTPUT_DIR}"; fi

config = load_yaml(REPO_DIR / "configs" / "collect_biot5_chebi20.yaml")
config["seed"] = int(SEED)
config["model"]["model_max_length"] = int(
    config["model"].get("model_max_length", config["generation"].get("max_length", 512))
)
config["data"]["train_file"] = str(CHEBI_PROCESSED_DIR / "train.jsonl")
config["data"]["staging_dir"] = str(COLLECTION_OUTPUT_DIR)
config["data"]["derived_train_file"] = str(DERIVED_TRAIN_FILE)

target_count = int(TARGET_MOLECULES_PER_DESCRIPTION)
config["generation"]["target_molecules_per_description"] = target_count
config["generation"]["num_return_sequences"] = target_count
num_beam_groups = int(config["generation"].get("num_beam_groups", 1))
num_beams = max(int(config["generation"].get("num_beams", target_count)), target_count)
if num_beam_groups > 1:
    num_beams = ((num_beams + num_beam_groups - 1) // num_beam_groups) * num_beam_groups
config["generation"]["num_beams"] = num_beams
if num_beam_groups > 1 and config["generation"]["num_beams"] % num_beam_groups != 0:
    raise ValueError(
        "Kaggle collection config requires num_beams to be divisible by num_beam_groups; "
        f"got {config['generation']['num_beams']} and {num_beam_groups}."
    )

config.setdefault("runtime", {})["description_offset"] = 0
config["runtime"]["max_descriptions"] = int(MAX_DESCRIPTIONS)
config["runtime"]["num_parts"] = int(NUM_PARTS)
config["runtime"]["part_index"] = int(PART_INDEX)
dump_yaml(config, TEMP_CONFIG_PATH)
print(f"Wrote config: {TEMP_CONFIG_PATH}")
print(TEMP_CONFIG_PATH.read_text(encoding="utf-8"))


In [ ]:
%cd {REPO_DIR}
!python scripts/collect_biot5_chebi20.py --config "{TEMP_CONFIG_PATH}" --num-parts {NUM_PARTS} --part-index {PART_INDEX}


In [ ]:
if SPLIT_OUTPUT_DIR.exists():
    shutil.rmtree(SPLIT_OUTPUT_DIR)

split_paths = ensure_grouped_split_files(
    DERIVED_TRAIN_FILE,
    SPLIT_OUTPUT_DIR,
    seed=SEED,
    validation_fraction=VALIDATION_FRACTION,
    test_fraction=TEST_FRACTION,
)
summary = read_json(COLLECTION_OUTPUT_DIR / "summary.json")
split_counts = {name: len(read_jsonl(path)) for name, path in split_paths.items()}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config_snapshot = {
    "train_file": str(CHEBI_PROCESSED_DIR / "train.jsonl"),
    "max_descriptions": int(MAX_DESCRIPTIONS),
    "target_molecules_per_description": int(TARGET_MOLECULES_PER_DESCRIPTION),
    "num_return_sequences": int(TARGET_MOLECULES_PER_DESCRIPTION),
    "num_beams": int(config["generation"]["num_beams"]),
    "num_beam_groups": int(config["generation"].get("num_beam_groups", 1)),
    "num_parts": int(NUM_PARTS),
    "part_index": int(PART_INDEX),
    "collection_output_dir": str(COLLECTION_OUTPUT_DIR),
    "derived_train_file": str(DERIVED_TRAIN_FILE),
    "split_output_dir": str(SPLIT_OUTPUT_DIR),
    "split_paths": {name: str(path) for name, path in split_paths.items()},
    "split_counts": split_counts,
    "validation_fraction": float(VALIDATION_FRACTION),
    "test_fraction": float(TEST_FRACTION),
    "requested_raw_candidates": int(MAX_DESCRIPTIONS) * int(TARGET_MOLECULES_PER_DESCRIPTION),
    "runtime": runtime_report,
    "huggingface_login_status": huggingface_login_status,
}
CONFIG_SNAPSHOT_PATH.write_text(json_dumps(config_snapshot), encoding="utf-8")

required_outputs = ensure_paths_exist({
    "collection_output_dir": COLLECTION_OUTPUT_DIR,
    "collection_summary": COLLECTION_OUTPUT_DIR / "summary.json",
    "derived_train_file": DERIVED_TRAIN_FILE,
    "split_train": split_paths["train"],
    "split_validation": split_paths["validation"],
    "split_test": split_paths["test"],
    "config_snapshot": CONFIG_SNAPSHOT_PATH,
})
artifact_dir, manifest = export_stage_artifacts(
    stage_name=STAGE_NAME,
    artifact_map={
        "post_training_processed/train_multimol.jsonl": DERIVED_TRAIN_FILE,
        "grouped_splits": SPLIT_OUTPUT_DIR,
        "collection_summary.json": COLLECTION_OUTPUT_DIR / "summary.json",
        "config_snapshot.json": CONFIG_SNAPSHOT_PATH,
    },
    metadata={
        "required_outputs": required_outputs,
        "config_path": str(TEMP_CONFIG_PATH),
        "summary": summary,
    },
)
artifact_zip_path = create_zip_archive(artifact_dir, ARTIFACT_ZIP_PATH)
print(json_dumps({
    "artifact_dir": str(artifact_dir),
    "artifact_zip_path": str(artifact_zip_path),
    "split_counts": split_counts,
    "summary": {
        "selected_descriptions": summary.get("selected_descriptions"),
        "descriptions_with_accepted_molecules": summary.get("descriptions_with_accepted_molecules"),
        "raw_candidates": summary.get("raw_candidates"),
        "accepted_candidates": summary.get("accepted_candidates"),
        "derived_examples": summary.get("derived_examples"),
        "target_molecules_per_description": summary.get("target_molecules_per_description"),
    },
    "manifest": manifest,
}))
